In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import os

os.listdir()

In [ ]:
data_dictionary = pd.read_csv('data_dictionary.csv')
hubs = pd.read_csv('hubs.csv')
complaints = pd.read_csv('complaints.csv')
drivers = pd.read_csv('drivers.csv')
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
incidents = pd.read_csv('incidents.csv')
vehicles = pd.read_csv('vehicles.csv')
deliveries = pd.read_csv('deliveries.csv')
app_events = pd.read_csv('app_events.csv')

In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
customers.isnull().sum()

In [ ]:
customers['loyalty_score'] = customers['loyalty_score'].fillna(customers['loyalty_score'].mean())
customers['preferred_channel'] = customers['preferred_channel'].fillna('Unknown')
customers.drop_duplicates(inplace=True)

customers.isnull().sum()

In [ ]:
deliveries.info()

In [ ]:
deliveries.isnull().sum()

In [ ]:
deliveries['delivery_completed_at'] = deliveries['delivery_completed_at'].fillna('Not Completed')

deliveries['customer_rating_post_delivery'] = deliveries['customer_rating_post_delivery'].fillna(
    deliveries['customer_rating_post_delivery'].mean()
)

deliveries.drop_duplicates(inplace=True)

deliveries.isnull().sum()

In [ ]:
orders.info()

In [ ]:
orders['booking_channel'] = orders['booking_channel'].fillna('Unknown')
orders.drop_duplicates(inplace=True)

orders.isnull().sum()

In [ ]:
complaints.info()

In [ ]:
complaints['compensation_amount'] = complaints['compensation_amount'].fillna(
    complaints['compensation_amount'].mean()
)

complaints.drop_duplicates(inplace=True)

complaints.isnull().sum()

In [ ]:
drivers.info()

In [ ]:
drivers['training_score'] = drivers['training_score'].fillna(
    drivers['training_score'].mean()
)

drivers.drop_duplicates(inplace=True)

drivers.isnull().sum()

In [ ]:
vehicles.info()

In [ ]:
vehicles['battery_health_pct'] = vehicles['battery_health_pct'].fillna(
    vehicles['battery_health_pct'].mean()
)

vehicles.drop_duplicates(inplace=True)

vehicles.isnull().sum()

In [ ]:
hubs.info()

In [ ]:
hubs.drop_duplicates(inplace=True)
hubs.isnull().sum()

In [ ]:
incidents.info()

In [ ]:
incidents['resolved_hours'] = incidents['resolved_hours'].fillna(
    incidents['resolved_hours'].mean()
)

incidents.drop_duplicates(inplace=True)

incidents.isnull().sum()

In [ ]:
app_events.info()

In [ ]:
app_events['order_id'] = app_events['order_id'].fillna('No Order')
app_events.drop_duplicates(inplace=True)
app_events.isnull().sum()

In [ ]:
delivery_orders = pd.merge(deliveries, orders, on='order_id')
delivery_orders.head()

In [ ]:
delivery_orders['pickup_zone'] = delivery_orders['pickup_zone'].replace({
    'CENTRAL':'Central',
    'Ctr':'Central',
    'AIRPORT':'Airport',
    'EAST':'East',
    'NORTH':'North',
    'SOUTH':'South',
    'WEST':'West',
    'north':'North',
    'Riverside':'RiverSide'
})

In [ ]:
complaint_delivery = pd.merge(complaints, delivery_orders, on='order_id')
complaint_delivery[['complaint_type', 'delivery_status']].head()

In [ ]:
driver_performance = pd.merge(delivery_orders, drivers, on='driver_id')
driver_performance.groupby('employment_type')['delivery_status'].value_counts()

In [ ]:
vehicle_analysis = pd.merge(delivery_orders, vehicles, on='vehicle_id')
vehicle_analysis.groupby('maintenance_status')['delivery_status'].value_counts()

In [ ]:
from pandasql import sqldf
# Query 1: Driver Performance (Average rating and total orders per driver)
query1 = """
SELECT driver_id, AVG(driver_rating) AS avg_rating, COUNT(order_id) AS total_orders
FROM driver_performance
GROUP BY driver_id
ORDER BY avg_rating DESC
LIMIT 10
"""
sqldf(query1)

In [ ]:
# Query 2: Customer Loyalty (Top 10 customers by loyalty score and order count)
query2 = """
SELECT customer_id, loyalty_score, COUNT(order_id) AS total_orders
FROM orders
JOIN customers USING (customer_id)
GROUP BY customer_id
ORDER BY loyalty_score DESC
LIMIT 10
"""
sqldf(query2)

In [ ]:
# Query 3: Financial Loss (Total compensation paid out per zone)
query3 = """
SELECT pickup_zone, SUM(compensation_amount) AS total_compensation
FROM complaint_delivery
GROUP BY pickup_zone
ORDER BY total_compensation DESC
"""
sqldf(query3)

In [ ]:
# Query 4: App Latency (Average API latency for failed vs successful events)
query4 = """
SELECT success_flag, AVG(api_latency_ms) AS avg_latency
FROM app_events
GROUP BY success_flag
"""
sqldf(query4)

In [ ]:
final_summary = {
    "Highest Failure Zone": "Central",
    "Main Complaint Type": "Delay",
    "Highest Route Overrides": "Airport",
    "Highest Risk Vehicles": "InRepair",
    "Driver Performance Ranking": "Updated via SQL",
    "Major Financial Risk": "Compensation Costs from Delays and Damage"
}
final_summary